In [126]:
import subprocess
import pkg_resources

def install_if_needed(package, version):
    try:
        pkg = pkg_resources.get_distribution(package)
        if pkg.version != version:
            raise pkg_resources.VersionConflict(pkg, version)
    except (pkg_resources.DistributionNotFound, pkg_resources.VersionConflict):
        subprocess.check_call(
            ["pip", "install", "-q", f"{package}=={version}"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

install_if_needed("langchain-core", "0.3.72")
install_if_needed("langchain-openai", "0.3.28")
install_if_needed("langchain-community", "0.3.27")
install_if_needed("unstructured", "0.18.11")
install_if_needed("langchain-chroma", "0.2.5")
install_if_needed("langchain-text-splitters", "0.3.9")
install_if_needed("pydantic", "2.11.9")

In [127]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [128]:
loader = UnstructuredHTMLLoader(file_path="data/mg-zs-warning-messages.html")
car_docs = loader.load()

In [129]:
import os

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=os.environ["OPENAI_API_KEY"])

In [130]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(car_docs)

print(len(split_docs))
print(split_docs[0].page_content)

18
Warning Message Procedure Cruise Control Fault Indicates that the cruise control system has detected a fault. Please consult an MG Authorised Repairer as soon as possible. Active Speed Limiter Fault Indicates that the active speed limit system has detected a fault. Contact an MG Authorised Repairer as soon as possible. Engine Coolant Temperature High High engine coolant temperature could result in severe damage. As soon as conditions permit, safely stop the vehicle and switch off the engine and


In [131]:
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings
)

In [132]:
retriever = vector_store.as_retriever()

In [133]:
prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below.\n"
    "If the answer is not in the context, say 'I don't know'.\n\n"
    "{context}\n\nQuestion: {question}"
)

In [134]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [135]:
response = rag_chain.invoke("The Gasoline Particular Filter Full warning has appeared. What does this mean and what should I do about it?")
answer = response.content
print(answer)

The Gasoline Particular Filter Full warning indicates that the gasoline particular filter is full. You should consult an MG Authorised Repairer as soon as possible.
